# 面试问题：怎样实现 ReAct Agent Loop 并防止死循环和成本失控？

可直接复述的回答：Agent Loop 必须是显式状态机，每轮记录 Thought、Action、Observation 和预算变化。模型只提出动作，执行器负责 schema、权限和超时。预算至少包含步数、token、工具成本和墙钟时间。循环检测不能只比较原始文本，而应对目标、动作、参数和关键观察构造稳定指纹。重复但没有信息增益时要改策略或升级人工。工具错误要区分可重试与永久失败。终止原因和完整轨迹必须可重放。

后续代码使用可读业务记录验证回答主线。实验结果均标注为教学实验，只用于解释机制，不代表真实线上收益。


## 1. 真实案例：支付延迟告警与输入预览

案例保留值班系统常见字段：时间、信号来源、指标和值。五条信号共同指向支付依赖延迟；它们是脱敏离线事件，不会访问真实监控系统。


In [1]:
signals05 = [  # 构造真实语义的故障观察流。
    {"time": "10:01", "source": "alert", "metric": "checkout_p95_ms", "value": 2400},  # 结账延迟告警。
    {"time": "10:02", "source": "metric", "metric": "payment_timeout_rate", "value": 0.18},  # 支付超时率升高。
    {"time": "10:02", "source": "log", "metric": "payment_504_count", "value": 83},  # 支付网关出现504。
    {"time": "10:03", "source": "deploy", "metric": "checkout_release", "value": "unchanged"},  # 结账服务没有新发布。
    {"time": "10:04", "source": "dependency", "metric": "payment_provider_status", "value": "degraded"},  # 外部支付依赖确认降级。
]  # 完成五条可读故障信号。
print("教学实验输入：支付延迟故障信号")  # 标识输入预览。
for signal05 in signals05:  # 逐条展示事件时间线。
    print(signal05)  # 输出监控或依赖信号。


教学实验输入：支付延迟故障信号
{'time': '10:01', 'source': 'alert', 'metric': 'checkout_p95_ms', 'value': 2400}
{'time': '10:02', 'source': 'metric', 'metric': 'payment_timeout_rate', 'value': 0.18}
{'time': '10:02', 'source': 'log', 'metric': 'payment_504_count', 'value': 83}
{'time': '10:03', 'source': 'deploy', 'metric': 'checkout_release', 'value': 'unchanged'}
{'time': '10:04', 'source': 'dependency', 'metric': 'payment_provider_status', 'value': 'degraded'}


## 2. Baseline（基线）：无预算的 ReAct 往返

朴素策略在 `read_logs` 和 `read_metrics` 之间反复切换，即使观察没有变化也继续执行。用八步固定轨迹展示工具成本和无效循环。


In [2]:
baseline_actions05 = ["read_logs", "read_metrics", "read_logs", "read_metrics", "read_logs", "read_metrics", "read_logs", "read_metrics"]  # 构造无终止条件的动作序列。
baseline_trace05 = []  # 收集朴素 Agent 的执行轨迹。
for step05, action05 in enumerate(baseline_actions05, start=1):  # 按顺序模拟八次工具调用。
    observation05 = "payment_504_and_timeout_unchanged"  # 返回没有信息增益的重复观察。
    baseline_trace05.append({"step": step05, "action": action05, "observation": observation05, "cost": 1.0})  # 保存动作、观察和成本。
baseline_cost05 = sum(item05["cost"] for item05 in baseline_trace05)  # 计算无预算循环总成本。
print("基线轨迹：step | action | observation")  # 输出朴素循环表头。
for item05 in baseline_trace05:  # 逐步展示重复行为。
    print(item05)  # 输出一个无信息增益步骤。
print("基线总成本", baseline_cost05)  # 展示死循环带来的成本。


基线轨迹：step | action | observation
{'step': 1, 'action': 'read_logs', 'observation': 'payment_504_and_timeout_unchanged', 'cost': 1.0}
{'step': 2, 'action': 'read_metrics', 'observation': 'payment_504_and_timeout_unchanged', 'cost': 1.0}
{'step': 3, 'action': 'read_logs', 'observation': 'payment_504_and_timeout_unchanged', 'cost': 1.0}
{'step': 4, 'action': 'read_metrics', 'observation': 'payment_504_and_timeout_unchanged', 'cost': 1.0}
{'step': 5, 'action': 'read_logs', 'observation': 'payment_504_and_timeout_unchanged', 'cost': 1.0}
{'step': 6, 'action': 'read_metrics', 'observation': 'payment_504_and_timeout_unchanged', 'cost': 1.0}
{'step': 7, 'action': 'read_logs', 'observation': 'payment_504_and_timeout_unchanged', 'cost': 1.0}
{'step': 8, 'action': 'read_metrics', 'observation': 'payment_504_and_timeout_unchanged', 'cost': 1.0}
基线总成本 8.0


## 3. 核心实现：状态指纹、信息增益与多重预算

指纹包含当前目标、动作和归一化观察；同一指纹第二次出现且没有新事实时停止自动循环。每轮同时扣减步骤和工具成本，停止后给出明确升级原因。


In [3]:
def fingerprint05(goal05, action05, observation05):  # 构造语义稳定的循环检测指纹。
    return (goal05, action05, observation05)  # 组合目标、动作和关键观察。
goal05 = "定位支付延迟根因"  # 冻结本次 Agent 的业务目标。
planned_actions05 = ["read_logs", "read_metrics", "read_logs", "check_dependency", "draft_incident"]  # 定义候选诊断动作。
observations05 = {"read_logs": "payment_504", "read_metrics": "timeout_rate_18pct", "check_dependency": "provider_degraded", "draft_incident": "incident_ready"}  # 模拟工具权威观察。
seen05 = set()  # 保存已经执行且无增益的状态指纹。
guarded_trace05 = []  # 收集带预算和循环检测的轨迹。
stop_reason05 = "completed"  # 初始化终止原因。
remaining_steps05 = 6  # 设置最大动作步数预算。
remaining_cost05 = 5.0  # 设置工具成本预算。
for action05 in planned_actions05:  # 逐个评估模型提出的动作。
    observation05 = observations05[action05]  # 获取当前工具的归一化观察。
    state_key05 = fingerprint05(goal05, action05, observation05)  # 计算当前状态指纹。
    if state_key05 in seen05:  # 检查动作与观察是否无变化重复。
        stop_reason05 = "cycle_without_information_gain"  # 标记检测到无效循环。
        guarded_trace05.append({"action": action05, "observation": observation05, "decision": "stop_cycle", "steps_left": remaining_steps05, "cost_left": remaining_cost05})  # 保存停止事件。
        break  # 立即停止继续消耗工具预算。
    if remaining_steps05 <= 0 or remaining_cost05 < 1.0:  # 检查多重预算是否足够。
        stop_reason05 = "budget_exhausted"  # 标记预算耗尽。
        break  # 在执行工具前停止。
    seen05.add(state_key05)  # 记录本次有意义的状态。
    remaining_steps05 -= 1  # 扣减一步动作预算。
    remaining_cost05 -= 1.0  # 扣减一次工具成本。
    guarded_trace05.append({"action": action05, "observation": observation05, "decision": "continue", "steps_left": remaining_steps05, "cost_left": remaining_cost05})  # 保存可重放执行事件。
print("受控 Agent 轨迹：action | observation | decision | budget")  # 输出核心状态轨迹表头。
for item05 in guarded_trace05:  # 展示每轮动作与预算变化。
    print(item05)  # 输出一个状态迁移事件。


受控 Agent 轨迹：action | observation | decision | budget
{'action': 'read_logs', 'observation': 'payment_504', 'decision': 'continue', 'steps_left': 5, 'cost_left': 4.0}
{'action': 'read_metrics', 'observation': 'timeout_rate_18pct', 'decision': 'continue', 'steps_left': 4, 'cost_left': 3.0}
{'action': 'read_logs', 'observation': 'payment_504', 'decision': 'stop_cycle', 'steps_left': 4, 'cost_left': 3.0}


## 4. 结果表与结果解读

受控循环在第三次动作发现 `read_logs + payment_504` 指纹重复，提前停止并升级，而不是继续烧掉预算。它没有自动得出根因，因为还未执行依赖检查；正确行为是承认停滞并改计划。


In [4]:
guarded_cost05 = 5.0 - remaining_cost05  # 计算循环检测前实际消耗的工具成本。
information_states05 = len(seen05)  # 统计获得的不同状态数量。
print("策略 | 执行事件数 | 不同状态数 | 成本 | 终止原因")  # 输出策略对照表头。
print("无预算基线", len(baseline_trace05), 2, baseline_cost05, "未终止")  # 展示无效循环结果。
print("预算+指纹", len(guarded_trace05), information_states05, guarded_cost05, stop_reason05)  # 展示受控循环结果。
print("结果解读：系统节省了后续调用，但必须把停滞原因交给重规划器或人工")  # 解释安全停止不等于任务完成。


策略 | 执行事件数 | 不同状态数 | 成本 | 终止原因
无预算基线 8 2 8.0 未终止
预算+指纹 3 2 2.0 cycle_without_information_gain
结果解读：系统节省了后续调用，但必须把停滞原因交给重规划器或人工


## 5. 失败案例与修正：原始文本变化绕过循环检测

如果只比较完整观察文本，时间戳变化会让相同事实看似不同。修正方法是提取归一化关键事实，再生成指纹；本例把两个不同时间戳的 504 日志归一为同一观察。


In [5]:
raw_observations05 = ["10:02 payment 504", "10:05 payment 504"]  # 构造文本不同但语义相同的观察。
raw_unique05 = len(set(raw_observations05))  # 演示原始文本比较认为它们不同。
normalized_observations05 = [text05.split(" ", 1)[1].replace(" ", "_") for text05 in raw_observations05]  # 去除时间戳并归一化关键事实。
normalized_unique05 = len(set(normalized_observations05))  # 统计归一化后的真实状态数。
print("失败行为：原始文本唯一数", raw_unique05, raw_observations05)  # 展示时间戳绕过循环检测。
print("修正行为：语义事实唯一数", normalized_unique05, normalized_observations05)  # 展示归一化后识别重复。


失败行为：原始文本唯一数 2 ['10:02 payment 504', '10:05 payment 504']
修正行为：语义事实唯一数 1 ['payment_504', 'payment_504']


## 6. 生产边界与轨迹合同

生产 Agent 还需要工具超时、取消、幂等、审批和秘密隔离。指纹字段要按任务设计，过粗会误杀合法重试，过细会漏掉循环；trace 需要脱敏并绑定模型、Prompt 和工具版本。


In [6]:
loop_contract05 = {"max_steps": 6, "max_tool_cost": 5.0, "cycle_threshold": 2, "fingerprint": ["goal", "action", "normalized_observation"], "fallback": "human_oncall"}  # 定义可审计循环策略。
print("Agent Loop 发布合同", loop_contract05)  # 展示预算和循环检测版本字段。
print("生产替换点：真实工具网关、墙钟超时、token预算、重规划器和脱敏trace存储")  # 说明教学字典与线上状态机的差距。


Agent Loop 发布合同 {'max_steps': 6, 'max_tool_cost': 5.0, 'cycle_threshold': 2, 'fingerprint': ['goal', 'action', 'normalized_observation'], 'fallback': 'human_oncall'}
生产替换点：真实工具网关、墙钟超时、token预算、重规划器和脱敏trace存储


## 7. 最小回归测试

断言只保护循环检测、预算节省和归一化反例。


In [7]:
assert len(signals05) >= 5  # 保证案例仍有完整故障信号。
assert stop_reason05 == "cycle_without_information_gain"  # 保证重复状态会明确终止。
assert guarded_cost05 < baseline_cost05  # 保证循环检测减少无效工具成本。
assert normalized_unique05 < raw_unique05  # 保证语义归一化能合并时间戳噪声。
print("最小回归测试通过：预算、循环指纹与失败归一化保持有效")  # 显示核心 Agent Loop 不变量已验证。


最小回归测试通过：预算、循环指纹与失败归一化保持有效
